# 05 - Generar MAPX con rasters directos desde paths del mosaic dataset

Este notebook prepara el MAPX de publicacion agregando cada imagen como raster directo, referido al path fisico registrado en el mosaic dataset.

La logica es:
- Leer todos los registros del mosaic dataset, excluyendo overviews (Name que inicia con ov).
- Extraer el path real de cada raster con ExportMosaicDatasetPaths.
- Calcular el label final de cada capa quitando CL_MLP_PAO_IF_Ortho_ y la extension .tif.
- Agregar cada TIFF directo dentro de Vuelos Drone PAO > Imagenes Drone.
- Ordenar las capas de la mas nueva a la mas vieja.
- Exportar solo MAPX, sin modificar el APRX original.


## 0. Configuracion


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import re
import sys

import arcpy
import pandas as pd

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'core').exists() and (candidate / 'flujo_geosupport_final' / 'settings.json').exists():
            return candidate
        if candidate.name.lower() == 'flujo_geosupport_final' and (candidate / 'settings.json').exists():
            parent = candidate.parent
            if (parent / 'core').exists():
                return parent
    raise RuntimeError('No se pudo resolver PROJECT_ROOT: debe existir core/ y flujo_geosupport_final/settings.json')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import flujo_geosupport_final.scripts.geosupport_flujo_completo as flujo

FINAL_DIR = PROJECT_ROOT / 'flujo_geosupport_final'
SETTINGS_PATH = FINAL_DIR / 'settings.json'
with SETTINGS_PATH.open('r', encoding='utf-8') as file_obj:
    SETTINGS = json.load(file_obj)
flujo.apply_flow_settings(SETTINGS)

OUTPUT_DIR = FINAL_DIR / 'outputs' / '05_generar_mapx_mosaico_completo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_MOSAIC_DATASET = flujo.PATH_MOSAIC_DATASET

# Fuente unica de inventario: mosaic dataset completo. La capa final en el MAPX queda referida al path raster extraido del mosaico.
WHERE_MOSAIC_MANUAL = None

# APRX base de trabajo. No se guarda ni se sobrescribe; solo se usa para construir y exportar el MAPX.
APRX_PATH = flujo.APRX_PATH
MAP_NAME = flujo.APRX_MAP_NAME
PARENT_GROUP_NAME = flujo.APRX_PARENT_GROUP_NAME
TARGET_GROUP_NAME = flujo.APRX_TARGET_GROUP_NAME

PREFIX = 'CL_MLP_PAO_IF_Ortho_'
RASTER_SUFFIXES = ('.tif', '.tiff')

ADD_ONLY_MISSING_TO_GROUP = True
SAVE_ORIGINAL_APRX = False
EXPORT_MAPX = True
MAPX_OUTPUT = OUTPUT_DIR / SETTINGS.get('mapx_output_name', 'CL MLP PAO 27 Imagenes Aereas PAO Image Server.mapx')

print('Settings:', SETTINGS_PATH)
print('Mosaic dataset:', PATH_MOSAIC_DATASET)
print('Fuente de paths: ExportMosaicDatasetPaths')
print('APRX base:', APRX_PATH)
print('Mapa:', MAP_NAME)
print('Grupo destino:', f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}')
print('Salida:', OUTPUT_DIR)
print('MAPX:', MAPX_OUTPUT)

## 1. Definir fuente unica desde el mosaic dataset


In [ ]:
names_source = 'mosaic_dataset_completo_paths_raster'
print('La preparacion leera el mosaic dataset completo y extraera los paths fisicos de raster.')
print('WHERE manual:', WHERE_MOSAIC_MANUAL)


## 2. Consultar registros del mosaic dataset y exportar paths


In [ ]:
def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def sql_in_clause(field_name, values, chunk_size=900):
    clean_values = [str(value).strip() for value in values if str(value).strip()]
    if not clean_values:
        return '1 = 0'
    field = arcpy.AddFieldDelimiters(PATH_MOSAIC_DATASET, field_name)
    chunks = []
    for start in range(0, len(clean_values), chunk_size):
        values_sql = ', '.join(sql_literal(value) for value in clean_values[start:start + chunk_size])
        chunks.append(f'{field} IN ({values_sql})')
    return ' OR '.join(f'({chunk})' for chunk in chunks)


where_mosaic = WHERE_MOSAIC_MANUAL
print('Where mosaic:')
print(where_mosaic if where_mosaic else '<sin filtro: todos los registros del mosaico>')

mosaic_fields = ['OBJECTID', 'Name', 'FechaAdqui']
with arcpy.da.SearchCursor(PATH_MOSAIC_DATASET, mosaic_fields, where_clause=where_mosaic) as cursor:
    dfmosaic = pd.DataFrame(cursor, columns=mosaic_fields)
mask = dfmosaic['Name'].str.match(r'^ov',na=False,case=False)
dfmosaic = dfmosaic[~mask]
dfmosaic = dfmosaic.sort_values('FechaAdqui', ascending=True).reset_index(drop=True)
print(f'Total registros encontrados en mosaico: {dfmosaic.shape[0]}')
display(dfmosaic.head())

In [ ]:
mosaic_path_table = 'in_memory/mosaic_paths_for_aprx'
if arcpy.Exists(mosaic_path_table):
    arcpy.management.Delete(mosaic_path_table)

arcpy.ExportMosaicDatasetPaths_management(
    in_mosaic_dataset=PATH_MOSAIC_DATASET,
    out_table=mosaic_path_table,
    export_mode='ALL',
    types_of_paths='RASTER',
)

path_cols = [field.name for field in arcpy.ListFields(mosaic_path_table)]
with arcpy.da.SearchCursor(mosaic_path_table, path_cols) as cursor:
    dfpath = pd.DataFrame(cursor, columns=path_cols)

print('Columnas paths:', list(dfpath.columns))
print(f'Total paths exportados: {dfpath.shape[0]}')
display(dfpath.head())

## 3. Preparar DataFrame con path real y nuevo label de capa


In [ ]:
def first_existing_column(df, candidates):
    lower_lookup = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        found = lower_lookup.get(candidate.lower())
        if found:
            return found
    return None


def short_image_name(value):
    name = Path(str(value)).name
    if name.startswith('tmp_'):
        name = name.replace('tmp_', '', 1)
    if name.startswith(PREFIX):
        name = name.replace(PREFIX, '', 1)
    lower = name.lower()
    for suffix in RASTER_SUFFIXES:
        if lower.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def image_date_sort_key(label):
    short = short_image_name(label)
    match = re.match(r'^(\d{2})_(\d{2})_(\d{2})_(.*)$', short)
    if not match:
        return (0, 0, 0, short.lower())
    yy, mm, dd, rest = match.groups()
    return (int(yy), int(mm), int(dd), rest.lower())


source_oid_col = first_existing_column(dfpath, ['SourceOID', 'SOURCEOID', 'Source_OID'])
path_col = first_existing_column(dfpath, ['Path', 'PATH', 'URI', 'Raster', 'RASTER'])
if not source_oid_col:
    raise ValueError(f'No se encontro columna SourceOID en tabla de paths. Columnas: {list(dfpath.columns)}')
if not path_col:
    raise ValueError(f'No se encontro columna de path raster en tabla de paths. Columnas: {list(dfpath.columns)}')

dfmerge = pd.merge(
    left=dfmosaic,
    right=dfpath,
    left_on='OBJECTID',
    right_on=source_oid_col,
    how='inner',
)

dfmerge = dfmerge.rename(columns={path_col: 'RasterPath'})
dfmerge['new_label'] = dfmerge['Name'].map(short_image_name)
dfmerge['sort_key'] = dfmerge['new_label'].map(image_date_sort_key)
dfmerge = dfmerge.sort_values('FechaAdqui', ascending=False).reset_index(drop=True)

missing_names = []

print(f'Total de registros mosaico: {dfmosaic.shape[0]}')
print(f'Total de registros merge path: {dfmerge.shape[0]}')
print('Nombres no encontrados en mosaico: 0 (no aplica, la fuente es el mosaico completo)')

preview_cols = ['OBJECTID', 'Name', 'FechaAdqui', 'RasterPath', 'new_label']
display(dfmerge[preview_cols].head())

In [ ]:
prepared_csv = OUTPUT_DIR / '01_mosaic_paths_prepared_for_aprx.csv'
missing_csv = OUTPUT_DIR / '02_names_missing_in_mosaic.csv'

dfmerge.to_csv(prepared_csv, index=False, encoding='utf-8-sig')
pd.DataFrame({'Name': missing_names}).to_csv(missing_csv, index=False, encoding='utf-8-sig')

print('Preparacion APRX:', prepared_csv)
print('No encontrados en mosaico:', missing_csv)

## 4. Utilidades para agregar al grupo y ordenar


In [ ]:
def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def image_keys(value):
    if value is None:
        return set()
    raw = str(value).strip()
    if not raw:
        return set()
    basename = Path(raw).name
    stem = Path(basename).stem
    short = short_image_name(raw)
    keys = {
        raw.lower(),
        basename.lower(),
        stem.lower(),
        short.lower(),
        f'{PREFIX}{short}'.lower(),
        f'{PREFIX}{short}.tif'.lower(),
        f'{PREFIX}{short}.tiff'.lower(),
        f'{short}.tif'.lower(),
        f'{short}.tiff'.lower(),
    }
    return {key for key in keys if key}


def safe_data_source(layer):
    try:
        return layer.dataSource
    except Exception:
        return None


def find_group(map_obj, group_name, parent_group=None):
    groups = [layer for layer in map_obj.listLayers() if layer.isGroupLayer]
    matches = []
    for group in groups:
        if group.name != group_name:
            continue
        if parent_group is not None:
            expected_prefix = layer_long_name(parent_group) + '\\'
            if not layer_long_name(group).startswith(expected_prefix):
                continue
        matches.append(group)
    if not matches:
        available = [layer_long_name(layer) for layer in groups]
        parent_text = f' bajo {layer_long_name(parent_group)}' if parent_group else ''
        raise ValueError(f"No se encontro grupo '{group_name}'{parent_text}. Grupos disponibles: {available}")
    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara {layer_long_name(matches[0])}")
    return matches[0]


def is_direct_child(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix = group_long_name + '\\'
    if not long_name.startswith(prefix):
        return False
    relative = long_name[len(prefix):]
    return '\\' not in relative


def direct_raster_children(map_obj, group_layer):
    return [
        layer for layer in map_obj.listLayers()
        if layer != group_layer and is_direct_child(layer, group_layer) and layer.isRasterLayer and not layer.isGroupLayer
    ]


def add_raster_directly_to_group(map_obj, group_layer, raster_path, target_name):
    before = {layer_long_name(layer) for layer in direct_raster_children(map_obj, group_layer)}
    temp_layer = map_obj.addDataFromPath(raster_path)
    temp_layer.name = target_name
    try:
        map_obj.addLayerToGroup(group_layer, temp_layer, 'TOP')
    finally:
        map_obj.removeLayer(temp_layer)

    after_layers = direct_raster_children(map_obj, group_layer)
    new_layers = [layer for layer in after_layers if layer_long_name(layer) not in before]
    if new_layers:
        new_layers[0].name = target_name
        return new_layers[0]

    target_keys = image_keys(target_name)
    for layer in after_layers:
        layer_keys = image_keys(layer.name)
        layer_keys.update(image_keys(safe_data_source(layer)))
        if layer_keys.intersection(target_keys):
            layer.name = target_name
            return layer
    return None


def move_to_top_inside_group(map_obj, group_layer, layer_to_move):
    children = direct_raster_children(map_obj, group_layer)
    if not children or children[0] == layer_to_move:
        return
    map_obj.moveLayer(children[0], layer_to_move, 'BEFORE')


def order_group_newest_first(map_obj, group_layer):
    ordered = sorted(direct_raster_children(map_obj, group_layer), key=lambda layer: image_date_sort_key(layer.name), reverse=True)
    for layer in reversed(ordered):
        move_to_top_inside_group(map_obj, group_layer, layer)


def normalize_group_layer_names(map_obj, group_layer):
    renamed = 0
    for layer in direct_raster_children(map_obj, group_layer):
        new_name = short_image_name(layer.name)
        new_name = re.sub(r'_normalizada$', '', new_name)
        
        if layer.name != new_name:
            print(f'Renombrando: {layer.name} -> {new_name}')
            layer.name = new_name
            renamed += 1
    return renamed

## 5. Agregar, renombrar y ordenar APRX


In [ ]:
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
aprx_results = []

aprx = arcpy.mp.ArcGISProject(APRX_PATH)
try:
    maps = aprx.listMaps(MAP_NAME)
    if not maps:
        raise ValueError(f'No se encontro el mapa: {MAP_NAME}')
    map_obj = maps[0]
    parent_group = find_group(map_obj, PARENT_GROUP_NAME)
    target_group = find_group(map_obj, TARGET_GROUP_NAME, parent_group)

    print(f'Grupo padre encontrado: {layer_long_name(parent_group)}')
    print(f'Grupo destino encontrado: {layer_long_name(target_group)}')

    existing_keys = set()
    for layer in direct_raster_children(map_obj, target_group):
        existing_keys.update(image_keys(layer.name))
        existing_keys.update(image_keys(safe_data_source(layer)))

    added_count = 0
    skipped_count = 0
    error_count = 0

    for _, row in dfmerge.iterrows():
        raster_path = str(row['RasterPath'])
        target_name = str(row['new_label'])
        source_name = str(row['Name'])
        item = {
            'Name': source_name,
            'RasterPath': raster_path,
            'new_label': target_name,
            'FechaAdqui': row.get('FechaAdqui'),
            'status': None,
            'message': None,
        }

        target_keys = image_keys(target_name)
        target_keys.update(image_keys(source_name))
        target_keys.update(image_keys(raster_path))
        if ADD_ONLY_MISSING_TO_GROUP and target_keys.intersection(existing_keys):
            item['status'] = 'skipped_existing'
            skipped_count += 1
            aprx_results.append(item)
            continue

        try:
            layer = add_raster_directly_to_group(map_obj, target_group, raster_path, target_name)
            if layer is None:
                item['status'] = 'added_not_confirmed'
                item['message'] = 'ArcPy agrego la capa pero no se pudo confirmar como hija directa del grupo'
                print(f'Advertencia: capa agregada no confirmada: {target_name}')
            else:
                item['status'] = 'added'
                item['message'] = layer_long_name(layer)
                print(f'Agregada: {target_name}')
            existing_keys.update(target_keys)
            added_count += 1
        except Exception as error:
            item['status'] = 'error'
            item['message'] = str(error)
            print(f'Error agregando {raster_path}: {error}')
            error_count += 1
        aprx_results.append(item)

    renamed_count = normalize_group_layer_names(map_obj, target_group)
    order_group_newest_first(map_obj, target_group)

    final_layers = direct_raster_children(map_obj, target_group)
    print('Orden final, primeras 25 capas:')
    for layer in final_layers[:25]:
        print(f'- {layer.name}')

    if SAVE_ORIGINAL_APRX:
        aprx.save()
        print('APRX original guardado')

    if EXPORT_MAPX:
        MAPX_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        map_obj.exportToMAPX(out_mapx=str(MAPX_OUTPUT))
        print('MAPX exportado:', MAPX_OUTPUT)

    print(f'Resumen APRX: agregadas={added_count}, omitidas={skipped_count}, renombradas={renamed_count}, errores={error_count}, total_grupo={len(final_layers)}')
finally:
    del aprx

aprx_results_df = pd.DataFrame(aprx_results)
display(aprx_results_df.head(50))

## 6. Guardar resultados


In [ ]:
summary_csv = OUTPUT_DIR / '00_summary.csv'
results_csv = OUTPUT_DIR / '02_aprx_add_from_mosaic_results.csv'
print(results_csv)
summary_rows = [
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'names_source', 'value': str(names_source)},
    {'metric': 'mosaic_dataset', 'value': PATH_MOSAIC_DATASET},
    {'metric': 'aprx_path', 'value': APRX_PATH},
    {'metric': 'map_name', 'value': MAP_NAME},
    {'metric': 'target_group', 'value': f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}'},
    {'metric': 'requested_names_count', 'value': 'no_aplica_mosaic_completo'},
    {'metric': 'mosaic_records_count', 'value': len(dfmosaic)},
    {'metric': 'merged_paths_count', 'value': len(dfmerge)},
    {'metric': 'missing_names_count', 'value': len(missing_names)},
    {'metric': 'save_original_aprx', 'value': SAVE_ORIGINAL_APRX},
    {'metric': 'mapx_output', 'value': str(MAPX_OUTPUT)},
]
if not aprx_results_df.empty and 'status' in aprx_results_df.columns:
    for status, count in aprx_results_df['status'].value_counts(dropna=False).items():
        summary_rows.append({'metric': f'aprx_status_{status}', 'value': int(count)})

pd.DataFrame(summary_rows).to_csv(summary_csv, index=False, encoding='utf-8-sig')
aprx_results_df.to_csv(results_csv, index=False, encoding='utf-8-sig')

print('Resumen:', summary_csv)
print('Resultados APRX:', results_csv)
print('Preparacion APRX:', prepared_csv)
print('No encontrados en mosaico:', missing_csv)